In [1]:
!pip install lifelines

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 409.1/409.1 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.3/117.3 kB 9.4 MB/s eta 0:00:00
  Created wheel for autograd-gamma: filename=autograd_gamma-0.5.0-py3-none-any.whl size=4030 sha256=43cd2270a400a7efce5618f286fc545421b2aaf3c9a39efa87ccb6681b98de18
  Stored in directory: /root/.cache/pip/wheels/50/37/21/0a719b9d89c635e89ff24bd93b862882ad675279552013b2fb
Successfully built autograd-gamma


In [ ]:
# ==============================================================================
# PROYECTO LMA: PREDICCIÓN DE SUPERVIVENCIA (LSTM + C-INDEX + SHAP INTEGRADO)
# ==============================================================================

import numpy as np
import tensorflow as tf
import logging
tf.get_logger().setLevel('ERROR') # Controlar la memoria
from tensorflow.keras.models import Model
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input, Concatenate, Masking
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import StratifiedKFold
from lifelines.utils import concordance_index
import shap
import matplotlib.pyplot as plt

# --- 1. CONFIGURACIÓN DE DATOS SIMULADOS ---
# Simula tus 3 hojas de Excel (Dinámicos, Fijos y Supervivencia/Eventos)
N_PACIENTES = 420
MAX_TOMAS = 17
N_VAR_DINAMICAS = 4
N_VAR_FIJAS = 4

print("Generando estructura de datos...")
datos_din_lista = [np.random.rand(np.random.randint(3, 18), N_VAR_DINAMICAS) for _ in range(N_PACIENTES)]
X_dinamicos = pad_sequences(datos_din_lista, maxlen=MAX_TOMAS, dtype='float32', padding='post', value=-1.0)
X_fijos = np.random.rand(N_PACIENTES, N_VAR_FIJAS)
y_evento = np.random.choice([0, 1], size=N_PACIENTES, p=[0.58, 0.42])
y_tiempo = np.random.uniform(1, 60, size=N_PACIENTES)

# --- 2. CONSTRUCCIÓN DE LA ARQUITECTURA ---
def construir_modelo_supervivencia():
    input_temp = Input(shape=(MAX_TOMAS, N_VAR_DINAMICAS), name="Analiticas")
    masked = Masking(mask_value=-1.0)(input_temp)
    lstm_out = LSTM(32)(masked)

    input_fijo = Input(shape=(N_VAR_FIJAS,), name="Datos_Fijos")
    fijo_out = Dense(16, activation='relu')(input_fijo)

    fusion = Concatenate()([lstm_out, fijo_out])
    x = Dense(16, activation='relu')(fusion)
    x = Dropout(0.2)(x)
    output = Dense(1, activation='sigmoid')(x)

    modelo = Model(inputs=[input_temp, input_fijo], outputs=output)
    modelo.compile(optimizer='adam', loss='binary_crossentropy')
    return modelo

# --- 3. VALIDACIÓN CRUZADA (10-FOLD) ---
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
resultados_c_index = []

early_stop = EarlyStopping(monitor='loss', patience=5, restore_best_weights=True)

print(f"\nIniciando 10-Fold Cross-Validation para {N_PACIENTES} pacientes...\n")

for vuelta, (train_idx, test_idx) in enumerate(skf.split(X_fijos, y_evento)):

    tf.keras.backend.clear_session()

    X_din_tr, X_din_te = X_dinamicos[train_idx], X_dinamicos[test_idx]
    X_fij_tr, X_fij_te = X_fijos[train_idx], X_fijos[test_idx]
    y_ev_tr, y_ev_te = y_evento[train_idx], y_evento[test_idx]
    y_tiempo_te = y_tiempo[test_idx]

    modelo_actual = construir_modelo_supervivencia()

    modelo_actual.fit(
        [X_din_tr, X_fij_tr], y_ev_tr,
        epochs=50, batch_size=32, verbose=0,
        callbacks=[early_stop]
    )

    scores_riesgo = modelo_actual.predict([X_din_te, X_fij_te], verbose=0).flatten()
    c_index = concordance_index(y_tiempo_te, -scores_riesgo, y_ev_te)
    resultados_c_index.append(c_index)

    print(f"Vuelta {vuelta + 1}/10 completada -> C-index: {c_index:.3f}")

# --- 4. RESULTADOS DE PREDICCIÓN ---
print("\n" + "="*50)
print(" RESULTADOS FINALES DE VALIDACIÓN CLÍNICA")
print("="*50)
print(f"C-index Medio: {np.mean(resultados_c_index):.3f} (Escala 0.5 a 1.0)")
print(f"Variabilidad (SD): {np.std(resultados_c_index):.3f}")
print("="*50)

# --- 5. EXPLICABILIDAD CLÍNICA (SHAP) CON TODOS LOS PACIENTES ---
print("\nIniciando cálculo de explicabilidad SHAP (Aviso: Este proceso puede tardar varios minutos)...")

# 1. TUS NOMBRES DE VARIABLES REALES (¡Modifica estas palabras con tus variables médicas!)
# Escribe aquí tus 4 variables dinámicas (las de las analíticas)
nombres_dinamicas_base = ["Copias_WT1", "Quimerismo", "mutaciones_postTMO", "Detección_HLADR"]

# Escribe aquí tus 4 variables fijas (los datos basales del paciente)
nombres_fijas_base = ["Edad", "Sexo", "Riesgo_Citogenetico", "Tipo_Acondicionamiento"]

# Generamos los nombres automáticos para cada "toma" en el tiempo
nombres_dinamicos = []
for toma in range(MAX_TOMAS):
    for var in nombres_dinamicas_base:
        nombres_dinamicos.append(f"{var}_Mes{toma}") # Puedes cambiar "Mes" por "Dia" o "Toma"

nombres_shap = nombres_dinamicos + nombres_fijas_base

# 2. FUNCIÓN ENVOLTORIO (Traductor para SHAP)
def wrapper_prediccion_shap(matriz_plana):
    corte_dinamico = MAX_TOMAS * N_VAR_DINAMICAS
    din_plano = matriz_plana[:, :corte_dinamico]
    fijos = matriz_plana[:, corte_dinamico:]
    din_3d = din_plano.reshape(-1, MAX_TOMAS, N_VAR_DINAMICAS)
    return modelo_actual.predict([din_3d, fijos], verbose=0).flatten()

# 3. PREPARAMOS LOS DATOS: 350 PARA CÁLCULO Y 70 PARA TEST
# Juntamos todos los pacientes de tu estudio
X_din_total_plano = X_dinamicos.reshape(N_PACIENTES, -1)
X_total_shap = np.hstack((X_din_total_plano, X_fijos))

# Separamos exactamente como has pedido: 350 y 70
X_train_shap = X_total_shap[:350] # Los 350 primeros para el background
X_test_shap = X_total_shap[350:420] # Los 70 restantes para el test

# 4. CALCULAMOS SHAP CON LOS 420 PACIENTES EN TOTAL
print("Configurando el explicador con 350 pacientes de base...")
explainer = shap.KernelExplainer(wrapper_prediccion_shap, X_train_shap)

print("Calculando el peso de cada variable en los 70 pacientes de test (¡Paciencia!)...")
# Extraemos los valores SHAP
shap_values = explainer.shap_values(X_test_shap)

# 5. GENERAR Y GUARDAR EL GRÁFICO FINAL
plt.figure(figsize=(12, 10)) # He hecho la imagen un poco más alta para que quepan bien los nombres
shap.summary_plot(shap_values, X_test_shap, feature_names=nombres_shap, max_display=15, show=False)
# Nota: max_display=15 hace que solo muestre las 15 variables más importantes de las 75 totales,
# para que el gráfico no sea un borrón de texto ilegible en tu Word.

plt.title("Importancia Clínica de Variables en la Predicción (SHAP)")
plt.tight_layout()
plt.savefig("grafico_SHAP_Final_TFG.png", dpi=300, bbox_inches='tight')
print("\n¡Éxito! Gráfico 'grafico_SHAP_Final_TFG.png' guardado y listo para el TFG.")
plt.show()